In [2]:
import os
import sys
import json
import numpy as np
import torch
import torchvision

sys.path.append('..')
from datasets.open_world_clad import OWCladDetection
from clad.detection.cladd import get_cladd_trainval, get_cladd_test

from args import ARGS

{'CLAD': ('Car', 'Truck', 'Tram', 'Cyclist', 'Tricycle', 'Pedestrian', 'unknown')}
[INFO] No Detectron installation found, continuing without.


## Object Detection Dataset

In [3]:
root = '../../data'

train_sets, val_sets = get_cladd_trainval(root, avalanche=False)
test_sets = get_cladd_test(root, avalanche=False)

In [4]:
print(len(train_sets), len(val_sets))
print(len(test_sets))

4 4
4


In [5]:
for train, val in zip(train_sets, val_sets):
    print("[Train]: {}, [Val]: {}".format(len(train), len(val)))

[Train]: 4470, [Val]: 497
[Train]: 1329, [Val]: 148
[Train]: 1479, [Val]: 165
[Train]: 524, [Val]: 59


OWDetection 클랫와 동일하게 만들기 위해 필요한 컴포넌트
* __getitem__() 에서 불러올 수 있도록 만든 attributes
    * self.images, self.imgids, self.annotations

OWDetection과 동일한 output 출력할 수 있도록 수정 완료!
    

1. 4개의 Task로 나누어진 데이터를 1개로 통합.
2. T1, T2, T3 에 맞춰 클래스를 나누어 학습 데이터 생성.
3. t1_task.txt, t2_task.txt, t3_task.txt 로 파일 나누어 생성.

In [6]:
# Task 1: "Car", "Truck", "Bus"
# Task 2: "Cyclist", "Tricycle"
# Task 3: "Pedestrian"

In [9]:
# train
train_t1_img_id = set()
train_t2_img_id = set()
train_t3_img_id = set()
for data in train_sets:
    for obj_id, obj in data.obj_annotations.items():
        if obj['category_id'] in [3, 4, 5]:
            train_t1_img_id.add(obj['image_id'])
        elif obj['category_id'] in [2, 6]:
            train_t2_img_id.add(obj['image_id'])
        elif obj['category_id'] in [1]:
            train_t3_img_id.add(obj['image_id'])
        else:
            raise ValueError(f"Unknown category_id: {obj['category_id']}")

# val
val_t1_img_id = set()
val_t2_img_id = set()
val_t3_img_id = set()
for data in val_sets:
    for obj_id, obj in data.obj_annotations.items():
        if obj['category_id'] in [3, 4, 5]:
            val_t1_img_id.add(obj['image_id'])
        elif obj['category_id'] in [2, 6]:
            val_t2_img_id.add(obj['image_id'])
        elif obj['category_id'] in [1]:
            val_t3_img_id.add(obj['image_id'])
        else:
            raise ValueError(f"Unknown category_id: {obj['category_id']}")

# test
test_img_id = set()
for data in test_sets:
    for obj_id, obj in data.obj_annotations.items():
        test_img_id.add(obj['image_id'])

print("Num of Task 1 data - [Train]: {}, [Val]: {}".format(len(train_t1_img_id), len(val_t1_img_id)))
print("Num of Task 2 data - [Train]: {}, [Val]: {}".format(len(train_t2_img_id), len(val_t2_img_id)))
print("Num of Task 3 data - [Train]: {}, [Val]: {}".format(len(train_t3_img_id), len(val_t3_img_id)))
print("Num of All Task data - [Test]: {}".format(len(test_img_id)))

Num of Task 1 data - [Train]: 4997, [Val]: 4997
Num of Task 2 data - [Train]: 3480, [Val]: 3480
Num of Task 3 data - [Train]: 2563, [Val]: 2563
Num of All Task data - [Test]: 9969


In [10]:
train_t1_img_id == val_t1_img_id

True

In [50]:
# create Task 1 image set text file
with open('clad/clad_t1_train.txt', 'w') as f:
    for img_id in train_t1_img_id:
        f.write(f"{img_id}\n")

# create Task 2 image set text file
with open('clad/clad_t2_train.txt', 'w') as f:
    for img_id in train_t2_img_id:
        f.write(f"{img_id}\n")

# create Task 3 image set text file
with open('clad/clad_t3_train.txt', 'w') as f:
    for img_id in train_t3_img_id:
        f.write(f"{img_id}\n")

# create Test image set text file
with open('clad/clad_all_task_test.txt', 'w') as f:
    for img_id in test_img_id:
        f.write(f"{img_id}\n")

In [ ]:
# train = OWCladDetection(root=root, ids=list(task_1_img_id), annot_file=os.path.join(root, 'SSLAD-2D', 'labeled', 'annotations', 'instance_train.json'))
# test = OWCladDetection(root=root, ids=list(test_img_id), annot_file=os.path.join(root, 'SSLAD-2D', 'labeled', 'annotations', 'instance_test.json'))

### category_id 변경 하기
* [1, 2, 3, 4, 5, 6] -> [0, 1, 2, 3, 4, 5]
* ["Pedestrian", "Cyclist", "Car", "Truck", "Tram", "Tricycle"] -> ["Car", "Truck", "Tram", "Cyclist", "Tricycle", "Pedestrian"]

In [18]:
# annotation files
train_annot_file = os.path.join(root, 'SSLAD-2D', 'labeled', 'annotations', 'instance_train.json')
val_annot_file = os.path.join(root, 'SSLAD-2D', 'labeled', 'annotations', 'instance_val.json')
test_annot_file = os.path.join(root, 'SSLAD-2D', 'labeled', 'annotations', 'instance_test.json')

with open(train_annot_file, "r") as f:
    train_annot_json = json.load(f)

with open(val_annot_file, "r") as f:
    val_annot_json = json.load(f)

with open(test_annot_file, "r") as f:
    test_annot_json = json.load(f)

print(f"[Train] images: {len(train_annot_json['images'])}, annotations: {len(train_annot_json['annotations'])}")
print(f"[Val] images: {len(val_annot_json['images'])}, annotations: {len(val_annot_json['annotations'])}")
print(f"[Test] images: {len(test_annot_json['images'])}, annotations: {len(test_annot_json['annotations'])}")

[Train] images: 5000, annotations: 41110
[Val] images: 5000, annotations: 37129
[Test] images: 10000, annotations: 69881


In [20]:
train_annot_json
val_annot_json
test_annot_json

{'categories': [{'supercategory': 'Pedestrian', 'id': 1, 'name': 'Pedestrian'},
  {'supercategory': 'Cyclist', 'id': 2, 'name': 'Cyclist'},
  {'supercategory': 'Car', 'id': 3, 'name': 'Car'},
  {'supercategory': 'Truck', 'id': 4, 'name': 'Truck'},
  {'supercategory': 'Tram', 'id': 5, 'name': 'Tram'},
  {'supercategory': 'Tricycle', 'id': 6, 'name': 'Tricycle'}],
 'annotations': [{'image_id': 1,
   'category_id': 4,
   'bbox': [691, 515, 35, 27],
   'area': 945,
   'id': 1,
   'truncated': -1,
   'occluded': -1,
   'iscrowd': 0},
  {'image_id': 1,
   'category_id': 4,
   'bbox': [761, 526, 21, 18],
   'area': 378,
   'id': 2,
   'truncated': -1,
   'occluded': -1,
   'iscrowd': 0},
  {'image_id': 1,
   'category_id': 4,
   'bbox': [897, 530, 20, 21],
   'area': 420,
   'id': 3,
   'truncated': -1,
   'occluded': -1,
   'iscrowd': 0},
  {'image_id': 1,
   'category_id': 4,
   'bbox': [809, 513, 40, 46],
   'area': 1840,
   'id': 4,
   'truncated': -1,
   'occluded': -1,
   'iscrowd': 0},

공통 변경 사항: supercategory

In [21]:
# 새로운 supercategory와 id 매핑
new_supercategories = ["Car", "Truck", "Tram", "Cyclist", "Tricycle", "Pedestrian"]
new_category_mapping = {name: idx for idx, name in enumerate(new_supercategories)}

# supercategory 변경 및 category_id 업데이트
data = train_annot_json

updated_categories = []
for category in data['categories']:
    supercategory = category['supercategory']
    if supercategory in new_category_mapping:
        updated_categories.append({
            'supercategory': supercategory,
            'id': new_category_mapping[supercategory],
            'name': category['name']
        })

print('Before:', [x['supercategory'] for x in data['categories']])
print('After:', new_supercategories)

new_category_mapping

Before: ['Pedestrian', 'Cyclist', 'Car', 'Truck', 'Tram', 'Tricycle']
After: ['Car', 'Truck', 'Tram', 'Cyclist', 'Tricycle', 'Pedestrian']


{'Car': 0, 'Truck': 1, 'Tram': 2, 'Cyclist': 3, 'Tricycle': 4, 'Pedestrian': 5}

각 파일 별로 supercategory 변경 및 category_id 업데이트
* train
* val
* test

In [22]:
# train 데이터 업데이트
train_data = train_annot_json
updated_annotations = []
for annotation in train_data['annotations']:
    # 기존 category_id에 해당하는 supercategory를 찾음
    old_category = next((cat for cat in train_data['categories'] if cat['id'] == annotation['category_id']), None)
    if old_category:
        new_category_id = new_category_mapping[old_category['supercategory']]
        updated_annotations.append({
            **annotation,
            'category_id': new_category_id
        })

# 최종 데이터
updated_train_data = {
    'categories': updated_categories,
    'annotations': updated_annotations,
    'images': train_data['images']
}
# 최종 데이터 저장
with open('clad/updated_instance_train.json', 'w') as f:
    json.dump(updated_train_data, f)

# val 데이터 업데이트
val_data = val_annot_json
updated_annotations = []
for annotation in val_data['annotations']:
    # 기존 category_id에 해당하는 supercategory를 찾음
    old_category = next((cat for cat in val_data['categories'] if cat['id'] == annotation['category_id']), None)
    if old_category:
        new_category_id = new_category_mapping[old_category['supercategory']]
        updated_annotations.append({
            **annotation,
            'category_id': new_category_id
        })

# 최종 데이터
updated_val_data = {
    'categories': updated_categories,
    'annotations': updated_annotations,
    'images': val_data['images']
}
# 최종 데이터 저장
with open('clad/updated_instance_val.json', 'w') as f:
    json.dump(updated_val_data, f)

# test 데이터 업데이트
test_data = test_annot_json
updated_annotations = []
for annotation in test_data['annotations']:
    # 기존 category_id에 해당하는 supercategory를 찾음
    old_category = next((cat for cat in test_data['categories'] if cat['id'] == annotation['category_id']), None)
    if old_category:
        new_category_id = new_category_mapping[old_category['supercategory']]
        updated_annotations.append({
            **annotation,
            'category_id': new_category_id
        })

# 최종 데이터
updated_test_data = {
    'categories': updated_categories,
    'annotations': updated_annotations,
    'images': test_data['images']
}
# 최종 데이터 저장
with open('clad/updated_instance_test.json', 'w') as f:
    json.dump(updated_test_data, f)

In [23]:
updated_val_data

{'categories': [{'supercategory': 'Pedestrian', 'id': 5, 'name': 'Pedestrian'},
  {'supercategory': 'Cyclist', 'id': 3, 'name': 'Cyclist'},
  {'supercategory': 'Car', 'id': 0, 'name': 'Car'},
  {'supercategory': 'Truck', 'id': 1, 'name': 'Truck'},
  {'supercategory': 'Tram', 'id': 2, 'name': 'Tram'},
  {'supercategory': 'Tricycle', 'id': 4, 'name': 'Tricycle'}],
 'annotations': [{'image_id': 1,
   'category_id': 3,
   'bbox': [1563, 633, 114, 174],
   'area': 19836,
   'id': 1,
   'truncated': -1,
   'occluded': -1,
   'iscrowd': 0},
  {'image_id': 1,
   'category_id': 5,
   'bbox': [1383, 666, 39, 48],
   'area': 1872,
   'id': 2,
   'truncated': -1,
   'occluded': -1,
   'iscrowd': 0},
  {'image_id': 1,
   'category_id': 3,
   'bbox': [1227, 638, 80, 99],
   'area': 7920,
   'id': 3,
   'truncated': -1,
   'occluded': -1,
   'iscrowd': 0},
  {'image_id': 1,
   'category_id': 3,
   'bbox': [1173, 635, 31, 75],
   'area': 2325,
   'id': 4,
   'truncated': -1,
   'occluded': -1,
   'isc

In [80]:
# check updated categories
print(f'[Train] {updated_train_data["categories"]}')
print(f'[Val] {updated_val_data["categories"]}')
print(f'[Test] {updated_test_data["categories"]}')

[Train] [{'supercategory': 'Pedestrian', 'id': 5, 'name': 'Pedestrian'}, {'supercategory': 'Cyclist', 'id': 3, 'name': 'Cyclist'}, {'supercategory': 'Car', 'id': 0, 'name': 'Car'}, {'supercategory': 'Truck', 'id': 1, 'name': 'Truck'}, {'supercategory': 'Tram', 'id': 2, 'name': 'Tram'}, {'supercategory': 'Tricycle', 'id': 4, 'name': 'Tricycle'}]
[Val] [{'supercategory': 'Pedestrian', 'id': 5, 'name': 'Pedestrian'}, {'supercategory': 'Cyclist', 'id': 3, 'name': 'Cyclist'}, {'supercategory': 'Car', 'id': 0, 'name': 'Car'}, {'supercategory': 'Truck', 'id': 1, 'name': 'Truck'}, {'supercategory': 'Tram', 'id': 2, 'name': 'Tram'}, {'supercategory': 'Tricycle', 'id': 4, 'name': 'Tricycle'}]
[Test] [{'supercategory': 'Pedestrian', 'id': 5, 'name': 'Pedestrian'}, {'supercategory': 'Cyclist', 'id': 3, 'name': 'Cyclist'}, {'supercategory': 'Car', 'id': 0, 'name': 'Car'}, {'supercategory': 'Truck', 'id': 1, 'name': 'Truck'}, {'supercategory': 'Tram', 'id': 2, 'name': 'Tram'}, {'supercategory': 'Tri

## CladDetection 데이터 클래스 테스트

In [ ]:
from datasets.open_world_clad import OWCladDetection
from args import ARGS
import os

In [ ]:
args = ARGS
args.data_root = "../data"
args.PREV_INTRODUCED_CLS = 0
args.CUR_INTRODUCED_CLS = 3
args.num_classes = 7

# Task 1 train data
t1_train = OWCladDetection(args, root=args.data_root, image_set='clad_t1_train', annot_file=os.path.join(args.data_root, 'SSLAD-2D', 'labeled', 'annotations', 'updated_instance_train.json'))
t1_val = OWCladDetection(args, root=args.data_root, image_set='clad_t1_train', annot_file=os.path.join(args.data_root, 'SSLAD-2D', 'labeled', 'annotations', 'updated_instance_val.json'))
t1_test = OWCladDetection(args, root=args.data_root, image_set='clad_all_task_test', annot_file=os.path.join(args.data_root, 'SSLAD-2D', 'labeled', 'annotations', 'updated_instance_test.json'))

In [65]:
t1_train[2][1], t1_val[2][1], t1_test[2][1]

({'image_id': tensor(3),
  'labels': tensor([0, 0, 0, 1, 1]),
  'area': tensor([4130,  320,  357,  999, 6072]),
  'boxes': tensor([[1070.,  672., 1140.,  731.],
          [1035.,  682., 1055.,  698.],
          [1002.,  676., 1019.,  697.],
          [ 952.,  663.,  979.,  700.],
          [ 854.,  622.,  923.,  710.]]),
  'orig_size': tensor([1920, 1080]),
  'size': tensor([1920, 1080]),
  'iscrowd': tensor([0, 0, 0, 0, 0])},
 {'image_id': tensor(3),
  'labels': tensor([2, 0]),
  'area': tensor([18920, 49572]),
  'boxes': tensor([[1235.,  603., 1455.,  689.],
          [ 806.,  602., 1049.,  806.]]),
  'orig_size': tensor([1920, 1080]),
  'size': tensor([1920, 1080]),
  'iscrowd': tensor([0, 0])},
 {'image_id': tensor(3),
  'labels': tensor([0, 6, 6, 6, 6, 6, 6, 0]),
  'area': tensor([474525,  10220,   7154,   6440,   7104,   3135,   6150, 114905]),
  'boxes': tensor([[ 921.,  604., 1920., 1079.],
          [ 849.,  629.,  919.,  775.],
          [ 702.,  614.,  751.,  760.],
        

In [ ]:
args.PREV_INTRODUCED_CLS = 3
args.CUR_INTRODUCED_CLS = 2

# Task 2 train data
t2_train = OWCladDetection(args, root=args.data_root, image_set='clad_t2_train', annot_file=os.path.join(args.data_root, 'SSLAD-2D', 'labeled', 'annotations', 'updated_instance_train.json'))
t2_val = OWCladDetection(args, root=args.data_root, image_set='clad_t2_train', annot_file=os.path.join(args.data_root, 'SSLAD-2D', 'labeled', 'annotations', 'updated_instance_val.json'))
t2_test = OWCladDetection(args, root=args.data_root, image_set='clad_all_task_test', annot_file=os.path.join(args.data_root, 'SSLAD-2D', 'labeled', 'annotations', 'updated_instance_test.json'))

In [ ]:
args.PREV_INTRODUCED_CLS = 5
args.CUR_INTRODUCED_CLS = 1

# Task 3 train data
t3_train = OWCladDetection(args, root=args.data_root, image_set='clad_t3_train', annot_file=os.path.join(args.data_root, 'SSLAD-2D', 'labeled', 'annotations', 'updated_instance_train.json'))
t3_val = OWCladDetection(args, root=args.data_root, image_set='clad_t3_train', annot_file=os.path.join(args.data_root, 'SSLAD-2D', 'labeled', 'annotations', 'updated_instance_val.json'))
t3_test = OWCladDetection(args, root=args.data_root, image_set='clad_all_task_test', annot_file=os.path.join(args.data_root, 'SSLAD-2D', 'labeled', 'annotations', 'updated_instance_test.json'))

In [80]:
print('[Train] - image_id: {}, file_name: {}, '.format(updated_train_data['images'][0]['id'], updated_train_data['images'][0]['file_name']))
print('[Val] - image_id: {}, file_name: {}, '.format(updated_val_data['images'][0]['id'], updated_val_data['images'][0]['file_name']))
print('[Test] - image_id: {}, file_name: {}, '.format(updated_test_data['images'][0]['id'], updated_test_data['images'][0]['file_name']))

[Train] - image_id: 1, file_name: HT_TRAIN_000001_SH_000.jpg, 
[Val] - image_id: 1, file_name: HT_VAL_000001_SH_001.jpg, 
[Test] - image_id: 1, file_name: HT_TEST_000001_SH_011.jpg, 


## Final check for the updated datasets

In [ ]:
from torch.utils.data import DataLoader
import util.misc as utils
import torch


sampler_train = torch.utils.data.RandomSampler(t1_train)
batch_sampler_train = torch.utils.data.BatchSampler(sampler_train, 4, drop_last=True)
data_loader_train = DataLoader(t1_train, batch_sampler=batch_sampler_train,
                                collate_fn=utils.collate_fn, num_workers=args.num_workers,
                                pin_memory=True)

In [82]:
data = next(iter(data_loader_train))
data

[tensor([[[[0.3843, 0.3843, 0.3843,  ..., 0.4784, 0.4784, 0.4784],
           [0.3804, 0.3804, 0.3804,  ..., 0.4784, 0.4784, 0.4784],
           [0.3765, 0.3765, 0.3765,  ..., 0.4784, 0.4784, 0.4784],
           ...,
           [0.1059, 0.1098, 0.1098,  ..., 0.1843, 0.1843, 0.1882],
           [0.1020, 0.1059, 0.1059,  ..., 0.1765, 0.1804, 0.1843],
           [0.0980, 0.1020, 0.1020,  ..., 0.1804, 0.1843, 0.1922]],
 
          [[0.6314, 0.6314, 0.6314,  ..., 0.6510, 0.6510, 0.6510],
           [0.6275, 0.6275, 0.6275,  ..., 0.6510, 0.6510, 0.6510],
           [0.6235, 0.6235, 0.6235,  ..., 0.6510, 0.6510, 0.6510],
           ...,
           [0.1098, 0.1137, 0.1137,  ..., 0.2039, 0.2039, 0.2078],
           [0.1059, 0.1098, 0.1098,  ..., 0.2000, 0.2039, 0.2078],
           [0.1020, 0.1059, 0.1059,  ..., 0.2039, 0.2078, 0.2157]],
 
          [[0.7686, 0.7686, 0.7686,  ..., 0.8431, 0.8431, 0.8431],
           [0.7647, 0.7647, 0.7647,  ..., 0.8431, 0.8431, 0.8431],
           [0.7608, 0.76